In [1]:
import numpy as np
from scipy import linalg

In [80]:
rng = np.random.RandomState(42)

Nsamp = 3000
n_param = 6

mu = np.zeros(n_param)
sigma = np.linspace(1,10,n_param)

max_signif = 10.0
theta = rng.rand(Nsamp,n_param)
param_mins = mu - max_signif*sigma
param_maxs = mu + max_signif*sigma
for p in range(n_param):
    theta[:,p] = param_mins[p] + theta[:,p]*(param_maxs[p]-param_mins[p])
loss = np.sum((theta - mu)**2/sigma**2,axis=1)
dLdtheta = 2*(theta - mu)/sigma**2

F_true = np.diag(1/sigma**2)
Cov_true = np.diag(sigma**2)
Sig_true = np.sqrt(np.diagonal(Cov_true))

prob_unif = np.exp(-0.5*loss)
F_unif = np.zeros((n_param,n_param),dtype=float)
for pa in range(n_param):
    for pb in range(pa,n_param):
        f = 0.25*np.sum(prob_unif*dLdtheta[:,pa]*dLdtheta[:,pb])/np.sum(prob_unif)
        F_unif[pa,pb] = f
        F_unif[pb,pa] = f
Cov_unif = linalg.inv(F_unif)
Sig_unif = 2*np.sqrt(np.sum(prob_unif)/np.sum(prob_unif*dLdtheta.T**2,axis=1))

N_iter = 20
C_scale = np.zeros(N_iter)
F_scale = np.zeros((N_iter,n_param,n_param),dtype=float)
Cov_scale = np.zeros_like(F_scale)
Sig_scale = np.zeros((N_iter,n_param),dtype=float)

max_signif_prev = 1.0
tol = 1e-2
i_break = -1
for i in range(N_iter):
    C_scale[i] = max_signif_prev#loss.mean()/max_signif_prev
    prob_scale = np.exp(-0.5*loss/C_scale[i])
    for pa in range(n_param):
        for pb in range(pa,n_param):
            f = 0.25*np.sum(prob_scale*dLdtheta[:,pa]*dLdtheta[:,pb])/np.sum(prob_scale)
            F_scale[i,pa,pb] = f
            F_scale[i,pb,pa] = f
    Cov_scale[i] = C_scale[i]*linalg.inv(F_scale[i])
    sig_est = np.sqrt(np.diagonal(Cov_scale[i]))
    Sig_scale[i] = 2*np.sqrt(C_scale[i]*np.sum(prob_scale)/np.sum(prob_scale*dLdtheta.T**2,axis=1))
    max_signif_approx = np.max((param_maxs-param_mins)/(2*Sig_scale[i]))
    if np.fabs(max_signif_approx/max_signif_prev-1) < tol:
        i_break = i
        break
    max_signif_prev = 1.0*max_signif_approx
    print(i,max_signif_approx,max_signif)
    
Sig_approx = np.median(np.fabs(loss)/np.fabs(dLdtheta.T + 1e-15),axis=1)
Sig_approx[Sig_approx > 1e15] = 0.0

print('\nUnif')
print('... full cov: [',','.join(['{0:.3f}'.format(p) for p in np.sqrt(np.diagonal(Cov_unif))/Sig_true]),']')
print('... diag: [',','.join(['{0:.3f}'.format(p) for p in Sig_unif/Sig_true]),']')
        
print('\n Scale')
for i in range(N_iter):
    print('... iteration = {0:d}, scale = {1:.2f}'.format(i,C_scale[i]))
    print('... ... full cov: [',','.join(['{0:.3f}'.format(p) for p in np.sqrt(np.diagonal(Cov_scale[i]))/Sig_true]),']')
    print('... ... diag: [',','.join(['{0:.3f}'.format(p) for p in Sig_scale[i]/Sig_true]),']')
    if i == i_break:
        break
        
print('\nApprox: [',','.join(['{0:.3f}'.format(p) for p in Sig_approx/Sig_true]),']')


0 17.61692590745785 10.0
1 9.814847050710343 10.0
2 10.89412588561819 10.0
3 10.728994457276222 10.0

Unif
... full cov: [ 51.423,44.204,71.259,117.942,75.380,90.846 ]
... diag: [ 0.568,0.707,3.148,1.000,5.934,0.764 ]

 Scale
... iteration = 0, scale = 1.00
... ... full cov: [ 51.423,44.204,71.259,117.942,75.380,90.846 ]
... ... diag: [ 0.568,0.707,3.148,1.000,5.934,0.764 ]
... iteration = 1, scale = 17.62
... ... full cov: [ 1.042,1.020,1.041,1.051,1.067,1.049 ]
... ... diag: [ 1.038,1.019,1.038,1.048,1.064,1.046 ]
... iteration = 2, scale = 9.81
... ... full cov: [ 0.995,0.922,0.990,0.992,1.040,1.006 ]
... ... diag: [ 0.983,0.918,0.982,0.985,1.033,0.999 ]
... iteration = 3, scale = 10.89
... ... full cov: [ 0.996,0.935,0.990,0.994,1.037,1.007 ]
... ... diag: [ 0.986,0.932,0.983,0.988,1.031,1.001 ]
... iteration = 4, scale = 10.73
... ... full cov: [ 0.995,0.933,0.989,0.993,1.037,1.007 ]
... ... diag: [ 0.985,0.930,0.983,0.987,1.031,1.001 ]

Approx: [ 19.722,19.322,19.585,19.493,19.52